# Session 4: causal epoch features

Prepare training and validation features using the frozen split and coverage rule.
Each row represents a completed 30-second epoch. No model is trained here.
The six feature columns contain sensor summaries only. Targets, coverage fields, and segment IDs are metadata.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sleep_classifier.features import FEATURE_COLUMNS
from sleep_classifier.prepare import prepare

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/prepare.yaml").exists())
output = prepare(ROOT / "configs/prepare.yaml")
provenance = json.loads((output / "provenance.json").read_text())
summary = pd.DataFrame(provenance["participants"])
display(summary.groupby("partition")[["epochs", "eligible_epochs"]].sum())

## Inspect the known development participant

Missing observations produce NaN summaries. Eligibility requires the selected coverage rule and a scored target.
Rejected rows receive segment ID -1. A subsequent valid row starts a new segment.
Segment IDs are local to each participant. Pair participant ID with segment ID when building sequences.


In [ ]:
table = pd.read_csv(output / "train/1066528.csv")
display(table.head())
print("Model inputs:", FEATURE_COLUMNS)
print("Retained epochs:", int(table.eligible.sum()), "of", len(table))

## Feature timelines

These values retain physical units. Heart rate uses bpm. Acceleration magnitude uses g and includes gravity.
Its standard deviation measures variation within the epoch; it is not a calibrated movement score.
No whole-night normalization or future interpolation is applied.


In [ ]:
valid = table.loc[table.eligible]
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].scatter(valid.time / 3600, valid.heart_rate_mean, s=4)
axes[0].set_ylabel("Mean heart rate (bpm)")
axes[1].scatter(valid.time / 3600, valid.motion_magnitude_std, s=4)
axes[1].set_ylabel("Magnitude std (g)")
axes[1].set_xlabel("Hours from PSG start")
plt.tight_layout()
plt.show()

## Next experiment

Fit a scaler on eligible training features only. Reuse it on validation and later test data.
Build a simple non-sequential classifier before the GRU. Use the same six inputs and participant split for both.
For recurrent models, construct sequences within each participant and segment. Reset recurrent state at every segment boundary.
Report prediction coverage alongside classification metrics. Final test observations remain unopened during preparation.
